# Bronze Layer Ingestion

This notebook implements the **Bronze layer** of the Medallion Architecture for the Observability Platform.

The Medallion Architecture organizes data into three layers:
- **Bronze (Raw):** Ingests raw data as-is from source systems with minimal transformation, preserving the original schema and fidelity.
- **Silver (Cleansed):** Applies cleaning, deduplication, type casting, and business logic to produce validated, queryable datasets.
- **Gold (Curated):** Aggregates and shapes data for direct consumption by dashboards, reports, and ML models.

This notebook reads raw Parquet files from Lakehouse Files (shortcuts to ADLS Gen2 containers) and writes them as Delta tables
in the Bronze layer. Sources include FOCUS-formatted cost exports, Azure Monitor metrics and logs, and Azure Resource Graph metadata.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, when, isnan, isnull, lit, current_timestamp, input_file_name
)
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType, LongType
)

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.parquet.mergeSchema", "true")

## 1. Ingest FOCUS Cost Data

Reads FOCUS-compliant cost export Parquet files from the `Files/costs/` shortcut.
Schema detection is enabled via `mergeSchema` to handle evolving export schemas across billing periods.

In [ ]:
df_costs_raw = (
    spark.read
    .format("parquet")
    .option("mergeSchema", "true")
    .option("recursiveFileLookup", "true")
    .load("Files/costs/")
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", input_file_name())
)

print(f"Cost records loaded: {df_costs_raw.count()}")
df_costs_raw.printSchema()

## 2. Ingest Metrics Data

Reads Azure Monitor metrics exported from Log Analytics workspaces in Parquet format.
These contain time-series metric values such as CPU utilization, CU consumption, and memory usage.

In [ ]:
df_metrics_raw = (
    spark.read
    .format("parquet")
    .option("mergeSchema", "true")
    .option("recursiveFileLookup", "true")
    .load("Files/metrics/")
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", input_file_name())
)

print(f"Metric records loaded: {df_metrics_raw.count()}")
df_metrics_raw.printSchema()

## 3. Ingest Log Data

Reads diagnostic and activity logs exported from Log Analytics in Parquet format.
These include operational events, errors, and audit trail entries from Azure services.

In [ ]:
df_logs_raw = (
    spark.read
    .format("parquet")
    .option("mergeSchema", "true")
    .option("recursiveFileLookup", "true")
    .load("Files/logs/")
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", input_file_name())
)

print(f"Log records loaded: {df_logs_raw.count()}")
df_logs_raw.printSchema()

## 4. Ingest Resource Graph Metadata

Reads Azure Resource Graph snapshots exported as Parquet files.
These contain resource properties, tags, SKU information, and hierarchical relationships.

In [ ]:
df_resource_metadata_raw = (
    spark.read
    .format("parquet")
    .option("mergeSchema", "true")
    .option("recursiveFileLookup", "true")
    .load("Files/metadata/")
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", input_file_name())
)

print(f"Resource metadata records loaded: {df_resource_metadata_raw.count()}")
df_resource_metadata_raw.printSchema()

## 5. Data Quality Checks

Before persisting to Delta, validate each dataset for null counts in key columns,
row counts, and schema completeness. This catches ingestion issues early.

In [ ]:
def run_quality_checks(df, table_name, required_columns):
    """Run data quality checks on a DataFrame before writing to Bronze."""
    row_count = df.count()
    print(f"\n--- Quality Report: {table_name} ---")
    print(f"Total rows: {row_count}")
    print(f"Total columns: {len(df.columns)}")
    print(f"Columns: {df.columns}")

    if row_count == 0:
        print(f"WARNING: {table_name} has zero rows. Skipping null checks.")
        return row_count

    present_required = [c for c in required_columns if c in df.columns]
    missing_required = [c for c in required_columns if c not in df.columns]

    if missing_required:
        print(f"WARNING: Missing expected columns: {missing_required}")

    if present_required:
        null_exprs = [
            count(when(isnull(col(c)), c)).alias(f"{c}_nulls")
            for c in present_required
        ]
        null_counts = df.select(null_exprs).collect()[0]
        for c in present_required:
            null_val = null_counts[f"{c}_nulls"]
            pct = (null_val / row_count) * 100
            status = "OK" if null_val == 0 else "WARN"
            print(f"  [{status}] {c}: {null_val} nulls ({pct:.1f}%)")

    return row_count


costs_count = run_quality_checks(
    df_costs_raw, "bronze_costs",
    ["BilledCost", "EffectiveCost", "ServiceName", "ChargePeriodStart"]
)
metrics_count = run_quality_checks(
    df_metrics_raw, "bronze_metrics",
    ["TimeGenerated", "MetricName", "MetricValue", "ResourceId"]
)
logs_count = run_quality_checks(
    df_logs_raw, "bronze_logs",
    ["TimeGenerated", "OperationName", "Category", "ResourceId"]
)
metadata_count = run_quality_checks(
    df_resource_metadata_raw, "bronze_resource_metadata",
    ["id", "name", "type", "resourceGroup", "subscriptionId"]
)

## 6. Write Bronze Delta Tables

Persist each raw dataset as a managed Delta table in the Lakehouse Tables area.
Using `overwrite` mode for full-refresh ingestion. For incremental patterns, switch to `append` with merge logic.

In [ ]:
bronze_tables = {
    "Tables/bronze_costs": df_costs_raw,
    "Tables/bronze_metrics": df_metrics_raw,
    "Tables/bronze_logs": df_logs_raw,
    "Tables/bronze_resource_metadata": df_resource_metadata_raw,
}

for table_path, df in bronze_tables.items():
    table_name = table_path.split("/")[-1]
    print(f"Writing {table_name} to {table_path}...")
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(table_path)
    )
    written_count = spark.read.format("delta").load(table_path).count()
    print(f"  {table_name}: {written_count} rows written successfully.")

print("\nBronze layer ingestion complete.")